# Launch synapse parameterization on the launch-system

A minimal end-to-end example that **registers** a synapse-parameterization config in entitycore
and then **launches it as a job on the launch-system** via the obi-one service, rather than
running the task in-process.

This mirrors `examples/app/test_task_launch_endpoint.ipynb` (which launches `circuit_extraction`),
but for the `circuit_synaptic_physiology_assignment` task type built in
`circuit_synapse_parameterization.ipynb`.

**Requirements**
- obi-one service running locally: `make run-local` (defaults to `http://127.0.0.1:8100`).
- The service must be configured with a reachable launch-system (`LAUNCH_SYSTEM_URL`).
- An access token copied from the platform and pasted below.

**The two-step shape.** The launch endpoint takes a *registered* config entity id, not an
in-memory config. So we first run a `GridScanGenerationTask` to register a campaign and its
single config(s) in entitycore, then POST the first single config's id to `/declared/task/launch`.

## Connect to the platform and the obi-one service

In [ ]:
from http import HTTPStatus

import httpx
import obi_one as obi
from entitysdk import Client, ProjectContext, models

environment = "staging"
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

# Paste a platform token here (same as test_task_launch_endpoint.ipynb)
token = "<PASTE-TOKEN>"

# entitycore DB client (used to register the config and read the execution activity back)
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(environment=environment, project_context=project_context, token_manager=token)

# obi-one service client (the API that submits jobs to the launch-system)
OBI_ONE_API_URL = "http://127.0.0.1:8100"
headers = {
    "Authorization": f"Bearer {token}",
    "virtual-lab-id": virtual_lab_id,
    "project-id": project_id,
}
api_client = httpx.Client(base_url=OBI_ONE_API_URL, headers=headers)

## Select the circuit

A small public circuit in staging keeps the job cheap. Any `CircuitFromID` works.

In [ ]:
from pathlib import Path

circuit_id = "8296b057-a0a0-41dc-86de-f64f6b8c0936"
circuit_from_id = obi.CircuitFromID(id_str=circuit_id)

circuit_entity = circuit_from_id.entity(db_client=db_client)
print(f"Circuit: {circuit_entity.name}  (id {circuit_entity.id}, scale {circuit_entity.scale})")

# Stage the circuit once (only to discover a sensible edge population); the launch-system job
# stages its own copy independently. You can also set `edge_population_name` by hand.
stage_dir = Path("../../../../../obi-output/synapse_parameterization_on_launch_system").resolve()
stage_dir /= f"inspect_cache/{circuit_entity.id}"
circuit = circuit_from_id.stage_circuit(
    db_client=db_client, dest_dir=stage_dir, entity_cache=True
)
sonata = circuit.sonata_circuit
try:
    edge_population_name = circuit.default_edge_population_name
except ValueError:
    intrinsic = [
        name
        for name in sonata.edges.population_names
        if sonata.edges[name].source.name == sonata.edges[name].target.name
    ]
    edge_population_name = (intrinsic or list(sonata.edges.population_names))[0]
print(f"Edge population to parameterize: {edge_population_name}")

## Build a simple scan config

Deliberately minimal: one excitatory synaptic model with a single conductance distribution, and
one `AllPairsSynapticModelAssigner` that applies it to every synapse in the edge population. Every
other synapse parameter falls back to its tagged default (this is exactly the default-resolution
machinery this PR adds).

We build a `ScanConfig` (not a `SingleConfig`) because the launch flow registers the config as a
campaign; a scan with no swept parameters yields a single coordinate.

In [ ]:
config = obi.SynapseParameterizationScanConfig.empty_config()

config.set(
    obi.Info(
        campaign_name="Synapse parameterization (launch demo)",
        campaign_description=f"Launch-system parameterization of circuit {circuit_entity.name}",
    ),
    name="info",
)
config.set(config.Initialize(circuit=circuit_from_id), name="initialize")

# One conductance distribution + one excitatory model referencing it
exc_gamma_dist = obi.GammaDistribution(shape=4.0, scale=0.25)
config.add(exc_gamma_dist, "Excitatory conductance distribution")

excitatory_model = obi.ExcitatoryTsodyksMarkramSynapticModel(
    conductance_distribution=exc_gamma_dist.ref
)
config.add(excitatory_model, "Excitatory synaptic model")

# Baseline assigner: every synapse in the edge population gets the excitatory model
all_pairs_assigner = obi.AllPairsSynapticModelAssigner(
    edge_population_name=edge_population_name,
    synaptic_model=excitatory_model.ref,
)
config.add(all_pairs_assigner, name="all_pairs")

# Resolve references (assigner -> model, model -> distribution) and assign block names
config.fill_block_references_and_names()
print("Distributions:", list(config.distributions.keys()))
print("Synaptic models:", list(config.synaptic_models.keys()))
print("Assigners:", list(config.synapse_model_assigners.keys()))

In [ ]:
# Validate config
config = config.validated_config()

## Register the config in entitycore

`GridScanGenerationTask.execute` registers the campaign and one single config per coordinate. With
no swept parameters there is exactly one. The launch endpoint needs the **single config's** entity
id.

In [ ]:
output_root = "../../../../../obi-output/synapse_parameterization_on_launch_system/grid_scan"
scan = obi.GridScanGenerationTask(
    form=config,
    coordinate_directory_option="ZERO_INDEX",
    output_root=output_root,
)
scan.execute(db_client=db_client)

campaign_entity = scan.form.campaign
print(f"Campaign '{campaign_entity.name}' (ID {campaign_entity.id})")
for cfg in scan.single_configs:
    print(f"  Coordinate {cfg.idx}: '{cfg.single_entity.name}' (ID {cfg.single_entity.id})")

## Launch the task on the launch-system

POST the first single config's id to `/declared/task/launch` with the task type
`circuit_synaptic_physiology_assignment`. The obi-one service reserves accounting, submits the job
to the launch-system, and returns the execution activity + job ids.

In [ ]:
single_config_entity = scan.single_configs[0].single_entity
single_config_id = single_config_entity.id
print(
    f"Selected '{single_config_entity.__class__.__name__}"
    f"({single_config_entity.task_config_type})' with ID {single_config_id}"
)

In [ ]:
payload = {
    "task_type": "circuit_synaptic_physiology_assignment",
    "config_id": str(single_config_id),
}
response = api_client.post(url="/declared/task/launch", json=payload, timeout=300.0)

if response.status_code == HTTPStatus.OK:
    data = response.json()
    print("Success:", data)
else:
    print(f"Error {response.status_code}: {response.text}")

## Inspect the execution activity

The activity carries the status, the config it used, the launch-system job that runs it, and
(once done) the registered parameterized circuit.

In [ ]:
execution_activity_id = data["activity_id"]
execution_activity = db_client.get_entity(
    entity_type=models.TaskActivity, entity_id=execution_activity_id
)
job_id = execution_activity.execution_id
print(
    f"Activity '{execution_activity.__class__.__name__}"
    f"({execution_activity.task_activity_type})': ID {execution_activity_id}"
)
print(f"  Status: {execution_activity.status}")
for _used in execution_activity.used:
    _used = db_client.get_entity(entity_id=_used.id, entity_type=models.TaskConfig)
    print(f"  Used '{_used.__class__.__name__}({_used.task_config_type})': ID {_used.id}")
for _gen in execution_activity.generated:
    print(f"  Generated '{_gen.type}': ID {_gen.id}")
print(f"  Executed by '{execution_activity.executor}': ID {job_id}")

## (Optional) Follow the job

The service proxies the launch-system job. Poll its status, or stream its log. Re-run the cell
above to refresh the activity status once the job finishes.

In [ ]:
job = api_client.get(url=f"/declared/task/{job_id}", timeout=60.0)
print(job.status_code, job.text)